# ICD-11 Knowledge Base — Chunking & Vector Database

This notebook builds the ICD-11 clinical knowledge base used as the RAG context source.  
It runs in two steps:

1. **Chunking** — extracts the ICD-11 CDDR PDF into structured JSON chunks (one chunk per disorder section)  
2. **Ingestion** — embeds the chunks with a biomedical model and stores them in ChromaDB

Once complete, the `icd11_clinical` collection is ready to query at inference time.


## Step 1 — PDF Chunking

Extracts the ICD-11 CDDR PDF into structured JSON.  
Each chunk corresponds to one section of one disorder (e.g. *Single Episode Depressive Disorder — Essential Features*).

**Output:** `../data/icd11_chunks.json`


In [ ]:
import re
import json
import subprocess
from pathlib import Path

# ── Paths (adjust if your layout differs) ─────────────────────────────────────
PDF_PATH    = r"../data/icd_11.pdf"
CHUNKS_PATH = r"../data/icd11_chunks.json"

# Pages to skip: front matter, TOC, list-of-categories
# Clinical content starts around page 70
CONTENT_START_PAGE = 70
CONTENT_END_PAGE   = 852

print("Config loaded.")
print(f"  PDF      : {PDF_PATH}")
print(f"  Output   : {CHUNKS_PATH}")


Config loaded.
  PDF      : C:\Users\Ramy\depression_suicide_detection\data\icd_11.pdf
  Output   : C:\Users\Ramy\depression_suicide_detection\data\icd11_chunks.json


In [37]:
# ── ICD-11 code → clinical domain mapping ─────────────────────────────────────
DOMAIN_MAP = {
    "6A0": "Neurodevelopmental disorders",
    "6A2": "Schizophrenia and other primary psychotic disorders",
    "6A4": "Catatonia",
    "6A6": "Mood disorders",
    "6A7": "Mood disorders",
    "6A8": "Mood disorders",
    "6B0": "Anxiety and fear-related disorders",
    "6B1": "Obsessive-compulsive and related disorders",
    "6B2": "Disorders specifically associated with stress",
    "6B4": "Dissociative disorders",
    "6B6": "Feeding and eating disorders",
    "6B8": "Elimination disorders",
    "6C0": "Disorders of bodily distress or experience",
    "6C2": "Disorders of bodily distress or experience",
    "6C4": "Disorders due to substance use or addictive behaviours",
    "6C5": "Disorders due to substance use or addictive behaviours",
    "6C6": "Disorders due to substance use or addictive behaviours",
    "6C7": "Disorders due to substance use or addictive behaviours",
    "6C9": "Impulse control disorders",
    "6D1": "Disruptive behaviour and dissocial disorders",
    "6D3": "Personality disorders",
    "6D4": "Paraphilic disorders",
    "6D5": "Factitious disorders",
    "6D6": "Neurocognitive disorders",
    "6D7": "Neurocognitive disorders",
    "6D8": "Neurocognitive disorders",
    "6E0": "Mental or behavioural disorders associated with pregnancy, childbirth or puerperium",
    "6E2": "Psychological and behavioural factors affecting health conditions",
    "6E4": "Secondary mental or behavioural syndromes",
    "6E6": "Secondary mental or behavioural syndromes",
}

def get_domain(code: str) -> str:
    for prefix, domain in DOMAIN_MAP.items():
        if code.startswith(prefix):
            return domain
    return "Other / Unclassified"

# ── Regex patterns ─────────────────────────────────────────────────────────────
SECTION_PATTERNS = [
    r"Essential \(required\) features",
    r"Essential features",
    r"Additional clinical features",
    r"Boundary with normality",
    r"Boundary with normality \(threshold\)",
    r"Course features",
    r"Developmental presentations",
    r"Culture-related features",
    r"Sex- and/or gender-related features",
    r"Boundaries with other disorders and conditions",
    r"Boundaries with other disorders and conditions \(differential diagnosis\)",
    r"Diagnostic requirements",
    r"Specifiers",
    r"Coded elsewhere",
    r"Note:",
]
SECTION_RE = re.compile(
    r"^(" + "|".join(SECTION_PATTERNS) + r")\s*$",
    re.IGNORECASE | re.MULTILINE,
)

# Matches lines like:  6A20   Schizophrenia
DISORDER_CODE_RE = re.compile(
    r"^(6[A-Z0-9]{2,5}(?:\.[A-Z0-9]{1,3})?)\s{2,}(.+)$",
    re.MULTILINE,
)

DISORDER_CODE_RE = re.compile(
    r"^(6[A-Z0-9]{2,5}(?:\.[A-Z0-9]{1,3})?)\s{2,}(.+)$",
    re.MULTILINE,
)

def normalise_section(heading: str) -> str:
    mapping = {
        "essential (required) features":                                     "Essential Features",
        "essential features":                                                "Essential Features",
        "additional clinical features":                                      "Additional Clinical Features",
        "boundary with normality":                                           "Boundary with Normality",
        "boundary with normality (threshold)":                               "Boundary with Normality",
        "course features":                                                   "Course Features",
        "developmental presentations":                                       "Developmental Presentations",
        "culture-related features":                                          "Culture-Related Features",
        "sex- and/or gender-related features":                               "Sex- and/or Gender-Related Features",
        "boundaries with other disorders and conditions":                    "Differential Diagnosis",
        "boundaries with other disorders and conditions (differential diagnosis)": "Differential Diagnosis",
        "diagnostic requirements":                                           "Diagnostic Requirements",
        "specifiers":                                                        "Specifiers",
    }
    return mapping.get(heading.strip().lower(), heading.strip())

def clean_line(line: str) -> str:
    line = line.strip()
    line = re.sub(
        r"^(Clinical Descriptions and Diagnostic Requirements for ICD-11 Mental.*|"
        r"Schizophrenia and other primary psychotic disorders \|.*|"
        r"[A-Z][a-z]+ (disorders?|syndrome|behaviour) \| .+)$",
        "", line, flags=re.IGNORECASE,
    )
    line = re.sub(r"^\d{1,4}\s*$", "", line)
    return line.strip()

print("Helpers and regex patterns ready.")


Helpers and regex patterns ready.


In [38]:
import shutil
import os

def extract_text_range(pdf_path: str, first: int, last: int) -> str:
    pdftotext_cmd = shutil.which("pdftotext")
    
    if pdftotext_cmd is None:
        conda_env = Path(os.environ.get("CONDA_PREFIX", ""))
        candidates = [
            conda_env / "Library" / "bin" / "pdftotext.exe",
            conda_env / "bin" / "pdftotext",
        ]
        for c in candidates:
            if c.exists():
                pdftotext_cmd = str(c)
                break

    if pdftotext_cmd is None:
        raise FileNotFoundError(
            "pdftotext not found. Run: conda install -c conda-forge poppler"
        )

    # Resolve to absolute path so pdftotext can find it regardless of cwd
    pdf_path = str(Path(pdf_path).resolve())
    print(f"Using pdftotext : {pdftotext_cmd}")
    print(f"PDF absolute path: {pdf_path}")
    print(f"File exists      : {Path(pdf_path).exists()}")

    result = subprocess.run(
        [pdftotext_cmd, "-f", str(first), "-l", str(last), "-layout", pdf_path, "-"],
        capture_output=True, text=True, encoding="utf-8", errors="replace",
    )

    if result.returncode != 0:
        print(f"pdftotext error: {result.stderr}")

    return result.stdout


def chunk_text(full_text: str) -> list:
    lines = full_text.split("\n")
    chunks = []
    current_disorder_code = None
    current_disorder_name = None
    current_section       = None
    current_lines         = []
    current_domain        = "Unclassified"

    def flush():
        nonlocal current_lines
        if not current_disorder_code or not current_section:
            current_lines = []
            return
        text = " ".join(l for l in current_lines if l).strip()
        text = re.sub(r"\s{2,}", " ", text)
        if len(text.split()) < 20:
            current_lines = []
            return
        chunks.append({
            "source":        "ICD-11 CDDR",
            "domain":        current_domain,
            "disorder_code": current_disorder_code,
            "disorder_name": current_disorder_name,
            "section":       current_section,
            "text":          text,
            "word_count":    len(text.split()),
        })
        current_lines = []

    for line in lines:
        cleaned = clean_line(line)
        if not cleaned:
            continue

        code_match = DISORDER_CODE_RE.match(cleaned)
        if code_match:
            flush()
            current_disorder_code = code_match.group(1)
            current_disorder_name = code_match.group(2).strip()
            current_domain        = get_domain(current_disorder_code)
            current_section       = "Overview"
            current_lines         = [current_disorder_name]
            continue

        sec_match = SECTION_RE.match(cleaned)
        if sec_match:
            flush()
            current_section = normalise_section(sec_match.group(1))
            continue

        current_lines.append(cleaned)

    flush()
    return chunks


def build_embed_text(chunk: dict) -> str:
    """
    Prepends structured metadata to the clinical text before embedding.
    This means the model encodes disorder + section context, not just raw text.
    """
    return (
        f"Source: {chunk['source']}\n"
        f"Domain: {chunk['domain']}\n"
        f"Disorder: {chunk['disorder_name']} ({chunk['disorder_code']})\n"
        f"Section: {chunk['section']}\n\n"
        f"{chunk['text']}"
    )


def postprocess(chunks: list, min_words: int = 40) -> list:
    """Merge consecutive near-empty chunks from the same disorder/section."""
    merged = []
    i = 0
    while i < len(chunks):
        c = chunks[i]
        while (
            c["word_count"] < min_words
            and i + 1 < len(chunks)
            and chunks[i + 1]["disorder_code"] == c["disorder_code"]
            and chunks[i + 1]["section"]       == c["section"]
        ):
            i += 1
            c = dict(c)
            c["text"]       = c["text"] + " " + chunks[i]["text"]
            c["word_count"] = len(c["text"].split())
        c["embed_text"] = build_embed_text(c)
        merged.append(c)
        i += 1
    return merged

print("Chunking functions defined.")


Chunking functions defined.


In [39]:
print(f"Extracting pages {CONTENT_START_PAGE}–{CONTENT_END_PAGE} from PDF …")
raw_text = extract_text_range(PDF_PATH, CONTENT_START_PAGE, CONTENT_END_PAGE)
print(f"  Extracted {len(raw_text):,} characters")

print("\nParsing into chunks …")
chunks = chunk_text(raw_text)
print(f"  Raw chunks : {len(chunks)}")

chunks = postprocess(chunks)
print(f"  After merge: {len(chunks)}")

# ── Stats ──────────────────────────────────────────────────────────────────────
domains  = {}
sections = {}
for c in chunks:
    domains [c["domain"] ] = domains.get (c["domain"],  0) + 1
    sections[c["section"]] = sections.get(c["section"], 0) + 1

print("\nChunks per domain:")
for d, n in sorted(domains.items(),  key=lambda x: -x[1]):
    print(f"  {n:>4}  {d}")

print("\nChunks per section:")
for s, n in sorted(sections.items(), key=lambda x: -x[1]):
    print(f"  {n:>4}  {s}")

# ── Save ───────────────────────────────────────────────────────────────────────
Path(CHUNKS_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(chunks)} chunks → {CHUNKS_PATH}")


Extracting pages 70–852 from PDF …
Using pdftotext : C:\Users\Ramy\anaconda3\envs\rag_for_mental_health\Library\bin\pdftotext.EXE
PDF absolute path: C:\Users\Ramy\depression_suicide_detection\data\icd_11.pdf
File exists      : True
  Extracted 2,521,469 characters

Parsing into chunks …
  Raw chunks : 1080
  After merge: 1080

Chunks per domain:
   151  Disorders due to substance use or addictive behaviours
   110  Mood disorders
   104  Secondary mental or behavioural syndromes
    99  Neurodevelopmental disorders
    84  Schizophrenia and other primary psychotic disorders
    80  Neurocognitive disorders
    68  Disorders specifically associated with stress
    60  Feeding and eating disorders
    59  Anxiety and fear-related disorders
    52  Elimination disorders
    48  Dissociative disorders
    46  Personality disorders
    37  Disorders of bodily distress or experience
    25  Impulse control disorders
    21  Disruptive behaviour and dissocial disorders
    15  Psychological a

In [40]:
# Inspect a sample chunk — check that embed_text looks right before ingesting
sample = next(c for c in chunks if c["section"] == "Essential Features" and c["word_count"] > 80)

print("── Chunk metadata ────────────────────────────────────────")
for k in ["domain", "disorder_code", "disorder_name", "section", "word_count"]:
    print(f"  {k:<16}: {sample[k]}")

print("\n── embed_text (first 600 chars) ──────────────────────────")
print(sample["embed_text"][:600])


── Chunk metadata ────────────────────────────────────────
  domain          : Neurodevelopmental disorders
  disorder_code   : 6A00
  disorder_name   : Disorders of intellectual development
  section         : Essential Features
  word_count      : 548

── embed_text (first 600 chars) ──────────────────────────
Source: ICD-11 CDDR
Domain: Neurodevelopmental disorders
Disorder: Disorders of intellectual development (6A00)
Section: Essential Features

• The presence of significant limitations in intellectual functioning across various domains such as perceptual reasoning, working memory, processing speed and verbal comprehension is required for diagnosis. There is often substantial variability in the extent to which any of these domains are affected in an individual. Whenever possible, performance should be measured using appropriately normed, standardized tests of intellectual functioning and found to


## Step 2 — ChromaDB Ingestion

Embeds each chunk using `FremyCompany/BioLORD-2023` and stores the vectors in a persistent ChromaDB collection.

**Why this model:**  
Trained on biomedical ontologies, UMLS concepts, and clinical term pairs.  
This means it understands clinical terminology and symptom descriptions on both sides — the ICD-11 clinical text and the social media posts — without needing to re-embed the social media collections.

**Output:** `../data/chroma_db/icd11_clinical` collection

In [42]:
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import torch

CHROMA_PATH      = "../data/chroma_db"
COLLECTION_NAME  = "icd11_clinical"
EMBEDDING_MODEL  = "FremyCompany/BioLORD-2023"
BATCH_SIZE       = 64

# ── Device ────────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("No GPU detected — using CPU")

# ── Load embedding model ───────────────────────────────────────────────────────
print(f"\nLoading: {EMBEDDING_MODEL} …")
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f"Loaded on  : {embedding_model.device}")
print(f"Output dims: {embedding_model.get_sentence_embedding_dimension()}")

# ── ChromaDB client ───────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
print(f"\nChromaDB path: {CHROMA_PATH}")


GPU : NVIDIA GeForce GTX 1650
VRAM: 4.29 GB

Loading: FremyCompany/BioLORD-2023 …


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\Ramy\anaconda3\envs\rag_for_mental_health\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ramy\.cache\huggingface\hub\models--FremyCompany--BioLORD-2023. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded on  : cuda:0
Output dims: 768

ChromaDB path: ../data/chroma_db


In [43]:
# Create the collection with cosine similarity
# (better than the default L2 for semantic text matching)
existing_names = [c.name for c in chroma_client.list_collections()]

if COLLECTION_NAME in existing_names:
    count = chroma_client.get_collection(COLLECTION_NAME).count()
    print(f"Collection '{COLLECTION_NAME}' already exists with {count} vectors.")
    print("To rebuild, set REBUILD = True below.")
    REBUILD = False   # ← set to True to wipe and re-ingest
    if REBUILD:
        chroma_client.delete_collection(COLLECTION_NAME)
        print("Deleted existing collection.")

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}   # cosine similarity space
)
print(f"Collection ready: '{COLLECTION_NAME}'  ({collection.count()} vectors)")


Collection ready: 'icd11_clinical'  (0 vectors)


In [44]:
# Load chunks from disk in case the kernel was restarted between steps
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)
print(f"Loaded {len(chunks)} chunks from {CHUNKS_PATH}")

if collection.count() == len(chunks):
    print("Collection already fully ingested. Skipping.")
else:
    print(f"\nIngesting {len(chunks)} chunks into '{COLLECTION_NAME}' …\n")

    for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Batches"):
        batch = chunks[i : i + BATCH_SIZE]

        # What we embed: metadata-prefixed text (disorder + section context baked in)
        texts_to_embed = [c["embed_text"] for c in batch]

        # Unique ID per chunk: code + section + position
        ids = [
            f"{c['disorder_code']}_{c['section'].replace(' ', '_').replace('/', '_')}_{i + j}"
            for j, c in enumerate(batch)
        ]

        # Metadata stored in Chroma — filterable at query time
        metadatas = [
            {
                "domain":        c["domain"],
                "disorder_code": c["disorder_code"],
                "disorder_name": c["disorder_name"],
                "section":       c["section"],
                "word_count":    c["word_count"],
            }
            for c in batch
        ]

        # Raw clinical text returned in query results (not the embed_text)
        documents = [c["text"] for c in batch]

        embeddings = embedding_model.encode(
            texts_to_embed,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            normalize_embeddings=True,  # required for cosine space
        ).tolist()

        collection.add(
            ids=ids,
            embeddings=embeddings,
            metadatas=metadatas,
            documents=documents,
        )

    print(f"\nDone. Collection '{COLLECTION_NAME}' now has {collection.count()} vectors.")


Loaded 1080 chunks from C:\Users\Ramy\depression_suicide_detection\data\icd11_chunks.json

Ingesting 1080 chunks into 'icd11_clinical' …



Batches: 100%|█████████████████████████████████████████████████████████████████████████| 17/17 [00:27<00:00,  1.62s/it]


Done. Collection 'icd11_clinical' now has 1080 vectors.


## Step 3 — Retrieval Functions

These are the functions you will import or copy into your main RAG evaluation notebook.

- `query_icd11()` — prints ranked results (for inspection / debugging)  
- `get_icd11_context()` — returns a formatted string ready to inject into an LLM prompt


In [52]:
def query_icd11(query_text: str, n_results: int = 5, section_filter: str = None):
    """
    Retrieve and print the top ICD-11 chunks for a given post.

    Args:
        query_text    : raw social media post text (no pre-cleaning needed)
        n_results     : number of chunks to return
        section_filter: restrict to one section type, e.g. 'Essential Features'
                        or 'Boundary with Normality'
    """
    query_embedding = embedding_model.encode(
        [query_text],
        normalize_embeddings=True,
    ).tolist()

    where = {"section": {"$eq": section_filter}} if section_filter else None

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    print(f"Query : '{query_text}'")
    if section_filter:
        print(f"Filter: section = '{section_filter}'")
    print("-" * 65)

    for i in range(len(results["documents"][0])):
        meta = results["metadatas"][0][i]
        doc  = results["documents"][0][i]
        dist = results["distances"][0][i]
        sim  = 1 - dist   # cosine distance → similarity

        print(f"[{i+1}] {meta['disorder_name']} ({meta['disorder_code']})")
        print(f"     Section    : {meta['section']}")
        print(f"     Domain     : {meta['domain']}")
        print(f"     Similarity : {sim:.4f}")
        print(f"     Text       : {doc}")
        print()


def get_icd11_context(
    query_text: str,
    n_results: int = 5,
    section_filter: str = None,
    include_metadata: bool = True,
) -> str:
    """
    Returns retrieved ICD-11 chunks as a formatted context string for LLM prompts.

    Args:
        query_text      : raw social media post
        n_results       : number of chunks to retrieve
        section_filter  : optional section filter (e.g. 'Essential Features')
        include_metadata: if True, each chunk is prefixed with disorder + section label
    
    Returns:
        A single string to inject into the RAG prompt context block.
    """
    query_embedding = embedding_model.encode(
        [query_text],
        normalize_embeddings=True,
    ).tolist()

    where = {"section": {"$eq": section_filter}} if section_filter else None

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where,
        include=["documents", "metadatas"],
    )

    parts = []
    for i in range(len(results["documents"][0])):
        meta = results["metadatas"][0][i]
        doc  = results["documents"][0][i]

        if include_metadata:
            header = (
                f"[{meta['disorder_name']} ({meta['disorder_code']}) "
                f"— {meta['section']}]"
            )
            parts.append(f"{header}\n{doc}")
        else:
            parts.append(doc)

    return "\n\n".join(parts)

print("Retrieval functions ready.")


Retrieval functions ready.


In [53]:
# ── Sanity-check queries ───────────────────────────────────────────────────────

# Test 1: general retrieval
query_icd11("I don't enjoy anything anymore and I feel completely worthless")


Query : 'I don't enjoy anything anymore and I feel completely worthless'
-----------------------------------------------------------------
[1] Mood disorder, unspecified (6A8Z)
     Section    : Additional Clinical Features
     Domain     : Mood disorders
     Similarity : 0.5127
     Text       : • In some individuals, the affective component of a depressive episode may be primarily experienced and expressed as irritability, or as an absence of emotional experience (e.g. “emptiness”). These variants in the expression of the affective component can be considered as meeting the depressed mood requirement for a depressive episode if they represent a significant change from the individual’s typical functioning. • In some individuals – particularly those experiencing a severe depressive episode – there may be reluctance to describe certain experiences (e.g. psychotic symptoms) or inability to do so in detail (e.g. due to psychomotor agitation or retardation). In such cases, observations m

In [54]:
# Test 2: filter to Essential Features only
query_icd11(
    "I don't want to wake up tomorrow",
    section_filter="Essential Features"
)


Query : 'I don't want to wake up tomorrow'
Filter: section = 'Essential Features'
-----------------------------------------------------------------
[1] Anxiety disorder induced by unknown or unspecified psychoactive substances (6C4G.71)
     Section    : Essential Features
     Domain     : Disorders due to substance use or addictive behaviours
     Similarity : 0.3900
     Text       : • The presentation is characterized by anxiety symptoms (e.g. apprehension or worry, fear, physiological symptoms of excessive autonomic arousal, panic attacks, avoidance behaviour) that develop during or soon after intoxication with or withdrawal from a specified substance, or use or discontinuation of a psychoactive medication. • The intensity or duration of the anxiety symptoms is substantially in excess of anxiety symptoms that are characteristic of intoxication or withdrawal due to the specified substance. • The specified substance, as well as the amount and duration of its use, is known to be capa

In [55]:
# Test 3: Boundary with Normality — useful for ambiguous posts
query_icd11(
    "I've been feeling really sad since my dog died last week",
    section_filter="Boundary with Normality"
)


Query : 'I've been feeling really sad since my dog died last week'
Filter: section = 'Boundary with Normality'
-----------------------------------------------------------------
[1] Prolonged grief disorder (6B42)
     Section    : Boundary with Normality
     Domain     : Dissociative disorders
     Similarity : 0.3971
     Text       : • An individual experiencing a grief reaction that is within a normative period given their cultural and religious context is considered to be experiencing normal bereavement, and should not be assigned a diagnosis of prolonged grief disorder. It is often important to consider whether other people who share the bereaved person’s cultural or religious perspective (e.g. family, friends, community) regard the response to the loss or duration of the reaction as abnormal. • Children and adolescents may respond to the loss of a primary attachment figure (e.g. a parent or caregiver) with an intense and sustained grief response (e.g. greater in intensity, sympt

In [56]:
# Test 4: show the formatted context string that goes into the LLM prompt
context = get_icd11_context(
    "I feel like everyone would be better off without me",
    n_results=3,
)
print("── LLM context block ─────────────────────────────────────────────────")
print(context)


── LLM context block ─────────────────────────────────────────────────
[Mood disorder, unspecified (6A8Z) — Additional Clinical Features]
• In some individuals, the affective component of a depressive episode may be primarily experienced and expressed as irritability, or as an absence of emotional experience (e.g. “emptiness”). These variants in the expression of the affective component can be considered as meeting the depressed mood requirement for a depressive episode if they represent a significant change from the individual’s typical functioning. • In some individuals – particularly those experiencing a severe depressive episode – there may be reluctance to describe certain experiences (e.g. psychotic symptoms) or inability to do so in detail (e.g. due to psychomotor agitation or retardation). In such cases, observations made by the clinician or reported by a collateral informant are important in determining the diagnostic status and severity of the episode. • Depressive episodes m

In [57]:
# Test 5: example of how to use get_icd11_context inside a RAG prompt
post = "I've been crying every day for two weeks and I can't get out of bed"
context = get_icd11_context(post, n_results=5)

prompt = f"""You are a clinical assistant helping assess mental health risk.
Use the ICD-11 clinical reference below to inform your response.

--- ICD-11 Clinical Reference ---
{context}
---------------------------------

Post: {post}

Based on the clinical reference, what disorder categories are most relevant to this post?
Explain your reasoning.""".strip()

print(prompt)


You are a clinical assistant helping assess mental health risk.
Use the ICD-11 clinical reference below to inform your response.

--- ICD-11 Clinical Reference ---
[Mood disorder, unspecified (6A8Z) — Essential Features]
• Both of the following symptoms occur concurrently and persist for most of the day, nearly every day, for at least several days. • Persistent elevation of mood or increased irritability is observable or reported, and represents a significant change from the individual’s usual range of moods (e.g. the change would be apparent to people who know the individual well). This does not include periods of elevated or irritable mood that are contextually appropriate (e.g. elevated mood after graduating from school or related to falling in love). Rapid shifts among different mood states commonly occur (i.e. mood lability). • Increased activity or a subjective experience of increased energy is present, and represents a significant change from the individual’s typical level. • In